In [2]:
from datetime import datetime, timedelta, date
import requests
import pandas as pd
import pickle
import numpy as np
import holidays

csv_te_voorspellen = './data csv/Input_voor_examen (1).csv'

In [3]:
def ophalenKijkcijferData(startDate, endDate):
  print(f"Ophalen kijkcijfer-data van {startDate.year}-{startDate.month}-{startDate.day} tot {endDate.year}-{endDate.month}-{endDate.day}")
  kijkcijfersData = []
  #elke dag ophalen (startDate is huidige dag)
  while startDate <= endDate:
    datum = f"{startDate.year}-{startDate.month}-{startDate.day}"
    url = f"https://api.cim.be/api/cim_tv_public_results_daily_views?dateDiff={datum}&reportType=north"

    try:
      response = requests.get(url)
      if response.status_code == 200:
        data = response.json()
        programmaLijst = data.get('hydra:member', [])
                        
        for programma in programmaLijst:
          try:
            kijkcijfersData.append({
              'dateDiff': programma.get('dateDiff'),
              'ranking': programma.get('ranking'),
              'description': programma.get('description'),
              'channel': programma.get('channel'),
              'startTime': programma.get('startTime'),
              'rLength': programma.get('rLength'),
              'rateInK': programma.get('rateInK'),
              'live': programma.get('live')
            })
                                   
          except Exception as e:
            print(f"error {datum}: {e}")         
      else:
        print(f"no data {datum}")
                        
    except Exception as e:
      print(f"error: {e}")
    
    startDate += timedelta(days=1)

  print("KijkcijferData opgehaald")
  df = pd.DataFrame(kijkcijfersData)

  return df

def ophalenWeerData(startDate, endDate):
    latitude = 51.05
    longitude = 3.7167
    today = datetime.today().date()

    hourly_vars = [
        "temperature_2m", "apparent_temperature", "weather_code", "precipitation",
        "rain", "snowfall", "cloud_cover", "windspeed_10m", "sunshine_duration"
    ]
    
    common_params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": hourly_vars,
        "timezone": "Europe/Brussels",
        "temperature_unit": "celsius",
        "precipitation_unit": "mm",
        "windspeed_unit": "kmh"
    }

    def fetch_weather_data(api_url, start, end):
        params = common_params.copy()
        params.update({
            "start_date": start.strftime('%Y-%m-%d'),
            "end_date": end.strftime('%Y-%m-%d')
        })
        response = requests.get(api_url, params=params)
        if response.status_code == 200:
            data = response.json().get("hourly", {})
            df = pd.DataFrame({var: data.get(var, []) for var in hourly_vars})
            df["timestamp"] = pd.to_datetime(data.get("time", []))
            if not df.empty:
                df["hour"] = df["timestamp"].dt.hour
                df["day_of_week"] = df["timestamp"].dt.dayofweek
                df["month"] = df["timestamp"].dt.month
                df["year"] = df["timestamp"].dt.year
            return df
        else:
            print(f"Fout bij ophalen data: {response.status_code}")
            print(response.text)
            return pd.DataFrame()

    dataframes = []

    # Historische data
    if startDate.date() < today:
        print("Ophalen historische data")
        hist_end = min(endDate.date(), today - timedelta(days=1))
        dataframes.append(fetch_weather_data(
            "https://archive-api.open-meteo.com/v1/archive",
            startDate, datetime.combine(hist_end, datetime.min.time())
        ))

    # Forecast data
    if endDate.date() >= today:
        print("Ophalen forecast data")
        forecast_start = max(endDate, datetime.combine(today, datetime.min.time()))
        print(forecast_start)
        dataframes.append(fetch_weather_data(
            "https://api.open-meteo.com/v1/forecast",
            forecast_start, endDate
        ))

    if dataframes:
        print("Weerdata opgehaald")
        return pd.concat(dataframes).sort_values("timestamp").reset_index(drop=True)
    else:
        return pd.DataFrame()


In [4]:
def testCSV(csv):
    df = pd.read_csv(csv, delimiter=';')
    # Kolommen hernoemen
    df = df.rename(columns={
        'Programma': 'description',
        'Zender': 'channel',
        'Datum': 'dateDiff',
        'Start': 'startTime',
        'Duur': 'rLength'
    })
    # Datum en tijd samenvoegen tot datetime
    df['dateDiff'] = pd.to_datetime(df['dateDiff'], dayfirst=True)
    # Starttijd naar HH:MM:SS
    df['startTime'] = df['startTime'].str[:8]
    # Duur naar HH:MM:SS
    df['rLength'] = df['rLength'].apply(lambda x: str(pd.to_timedelta(x)))
    # Voeg dummy kolommen toe als nodig
    df['ranking'] = 0
    df['live'] = 0
    df['Kijkers'] = None

    return df[['dateDiff', 'ranking', 'description', 'channel', 'startTime', 'rLength', 'live']]


In [5]:
teVoorspellen = testCSV(csv_te_voorspellen)

teVoorspellen['dateDiff'] = pd.to_datetime(teVoorspellen['dateDiff'])

end_date = teVoorspellen['dateDiff'].max()
start_date = end_date - timedelta(weeks=3)
histKijkcijfers = ophalenKijkcijferData(start_date, end_date - timedelta(days=1))
histWeerdata = ophalenWeerData(start_date, end_date)

Ophalen kijkcijfer-data van 2025-5-16 tot 2025-6-5
KijkcijferData opgehaald
Ophalen historische data
Ophalen forecast data
2025-06-06 00:00:00
Weerdata opgehaald


In [6]:
def cleanKijkcijferData(df):

    # Zet 'Kijkers' kolom, als 'rateInK' bestaat
    if 'rateInK' in df.columns:
        df['Kijkers'] = (
            df['rateInK']
            .dropna()
            .astype(str)
            .str.replace('.', '', regex=False)
            .astype(int)
        )
    else:
        df['Kijkers'] = None

    # rLength aanpassen
    df['rLength'] = df['rLength'].astype(str).apply(
    lambda x: x[-8:] if 'days' in x else x.zfill(8)
    )

    # Tijd aanpassen
    tijd_regex = r'^\d{2}:\d{2}:\d{2}$'
    # Omzetten naar datetime
    df['date'] = pd.to_datetime(df['dateDiff']).dt.date
    # Filter rijen met formaat
    df = df[df['startTime'].str.match(tijd_regex, na=False) & df['rLength'].str.match(tijd_regex, na=False)].copy()
    
    # Afleveringlengte naar seconden omzetten 
    df['Lengte_sec'] = pd.to_timedelta(df['rLength']).dt.total_seconds().astype(int)

    # Uren met 24+
    def time_cor(rij):
        tijdArr = rij['startTime'].split(':')
        if int(tijdArr[0]) >= 24:
            tijdArr[0] = str(int(tijdArr[0]) - 24).zfill(2)
            rij['date'] += timedelta(days=1)
        rij['startTime'] = ':'.join(tijdArr)
        return rij
    
    df = df.apply(time_cor, axis=1)

    # 1 kolom voor beide data
    df['FullDate'] = pd.to_datetime(df['date'].astype(str) 
                                    + " " + df['startTime'].astype(str))
    
    # Hour en minute voor join later on
    df['hour'] = pd.to_datetime(df['startTime'], format='%H:%M:%S').dt.hour
    df['minute'] = 0

    # Kolommen verwijderen die niet nodig meer zijn, als ze bestaan
    columns_to_drop = ['startTime', 'rLength', 'rateInK', 'ranking', 'live']
    df.drop([col for col in columns_to_drop if col in df.columns], axis=1, inplace=True)


    # De nieuwe dataframe
    df = df[['FullDate', 'date', 'hour', 'minute', 'channel', 'description', 'Lengte_sec', 'Kijkers']]

    # Hernoemen kolommen
    df.rename(columns={'description': 'Programma', 'channel': 'Kanaal'}, inplace=True)

    return df


def cleanWeerData(df):
  weerData = df
  weerData['timestamp'] = pd.to_datetime(weerData['timestamp'])
  #naar zelfde formaat als kijkcijfer datum
  weerData['datetime'] = weerData['timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S')

  #hour voor join later on
  weerData['hour'] = pd.to_datetime(weerData['datetime']).dt.hour
  weerData['minute'] = pd.to_datetime(weerData['datetime']).dt.minute
  weerData['date'] = pd.to_datetime(weerData['datetime']).dt.date

  #verwijder kolom
  weerData = weerData.drop(columns=['timestamp'])

  weerData = weerData[['datetime', 'date' ,'hour', 'minute', 'temperature_2m', 'apparent_temperature', 
                            'rain', 'snowfall', 'weather_code', 'cloud_cover', 
                            'windspeed_10m', 'sunshine_duration']]

  #hernoemen kolommen
  weerData.rename(columns={'temperature_2m':'Temperatuur', 'apparent_temperature':'Gevoelstemp', 'windspeed_10m': 'Windsnelheid', 'rain':'Regen', 'snowfall': 'Sneeuw', 'weather_code':'Weercode', 'cloud_cover':'Bewolking', 'sunshine_duration':'Zonnenschijn'}, inplace=True)

  return weerData

In [19]:
histWeerdataClean = cleanWeerData(histWeerdata)
histKijkcijfersClean = cleanKijkcijferData(histKijkcijfers)
teVoorspellenClean = cleanKijkcijferData(teVoorspellen)
display(histWeerdataClean)
display(histKijkcijfersClean)
display(teVoorspellenClean)

,datetime,date,hour,minute,Temperatuur,Gevoelstemp,Regen,Sneeuw,Weercode,Bewolking,Windsnelheid,Zonnenschijn
0,2025-05-16 00:00:00,2025-05-16,0,0,11.6,8.6,0.0,0.0,0.0,0.0,12.7,0.00
1,2025-05-16 01:00:00,2025-05-16,1,0,11.1,8.0,0.0,0.0,0.0,0.0,12.1,0.00
2,2025-05-16 02:00:00,2025-05-16,2,0,10.3,7.2,0.0,0.0,0.0,0.0,11.5,0.00
3,2025-05-16 03:00:00,2025-05-16,3,0,9.1,6.2,0.0,0.0,0.0,0.0,10.5,0.00
4,2025-05-16 04:00:00,2025-05-16,4,0,8.4,5.7,0.0,0.0,0.0,0.0,10.0,0.00
5,2025-05-16 05:00:00,2025-05-16,5,0,7.7,5.3,0.0,0.0,0.0,0.0,8.7,0.00
6,2025-05-16 06:00:00,2025-05-16,6,0,7.1,4.8,0.0,0.0,0.0,0.0,8.6,0.00
7,2025-05-16 07:00:00,2025-05-16,7,0,7.6,5.2,0.0,0.0,0.0,0.0,9.0,1690.92
8,2025-05-16 08:00:00,2025-05-16,8,0,9.4,7.3,0.0,0.0,0.0,0.0,7.4,3600.00
9,2025-05-16 09:00:00,2025-05-16,9,0,11.7,9.3,0.0,0.0,0.0,0.0,8.7,3600.00


,FullDate,date,hour,minute,Kanaal,Programma,Lengte_sec,Kijkers
0,2025-05-16 20:15:38,2025-05-16,20,0,VRT 1,THUIS,1489,937648
1,2025-05-16 19:00:05,2025-05-16,19,0,VRT 1,HET 7 UUR-JOURNAAL,2685,760119
2,2025-05-16 19:46:40,2025-05-16,19,0,VRT 1,IEDEREEN BEROEMD,1331,581626
3,2025-05-16 20:43:22,2025-05-16,20,0,VTM,I CAN SEE YOUR VOICE,5500,527119
4,2025-05-16 20:08:26,2025-05-16,20,0,VTM,FAMILIE,1468,526533
5,2025-05-16 18:28:54,2025-05-16,18,0,VRT 1,BLOKKEN,1708,496405
6,2025-05-16 18:59:48,2025-05-16,18,0,VTM,NIEUWS 19U VTM,3334,423571
7,2025-05-16 20:42:52,2025-05-16,20,0,VRT 1,LA VIE EN ROSE,2832,411246
8,2025-05-16 13:00:04,2025-05-16,13,0,VRT 1,HET 1 UUR-JOURNAAL,1744,330006
9,2025-05-16 21:30:46,2025-05-16,21,0,VRT CANVAS,RIDLEY,5310,320467


,FullDate,date,hour,minute,Kanaal,Programma,Lengte_sec,Kijkers
0,2025-06-06 20:17:52,2025-06-06,20,0,VRT 1,THUIS,1470,None
1,2025-06-06 19:00:04,2025-06-06,19,0,VRT 1,HET 7 UUR-JOURNAAL,2598,None
2,2025-06-06 18:29:07,2025-06-06,18,0,VRT 1,BLOKKEN,1710,None
3,2025-06-06 18:59:48,2025-06-06,18,0,VTM,NIEUWS 19U VTM,3201,None
4,2025-06-06 21:37:08,2025-06-06,21,0,VRT 1,DE DAG VAN VANDAAG,3206,None
5,2025-06-06 20:05:28,2025-06-06,20,0,VTM,FAMILIE,1544,None
6,2025-06-06 20:41:26,2025-06-06,20,0,VTM,HUIS GEMAAKT,3945,None
7,2025-06-06 13:00:04,2025-06-06,13,0,VRT 1,HET 1 UUR-JOURNAAL,1653,None


In [8]:
def mergen(kijkcijfers, weer):
  kijkcijfersWeer = pd.merge(kijkcijfers, weer, on=['date', 'hour'], how='left')
  kijkcijfersWeer = kijkcijfersWeer[['FullDate', 'date', 'hour', 'Kanaal', 'Programma', 'Lengte_sec', 'Kijkers', 'Temperatuur', 'Gevoelstemp', 'Regen', 'Sneeuw', 'Weercode', 'Bewolking', 'Windsnelheid', 'Zonnenschijn']]
  kijkcijfersWeer.dropna(inplace=True)
  return kijkcijfersWeer

In [9]:
histKijkcijfersWeer = mergen(histKijkcijfersClean, histWeerdataClean)
print(histKijkcijfersWeer)

               FullDate        date  hour                  Kanaal  \
0   2025-05-16 20:15:38  2025-05-16    20                   VRT 1   
1   2025-05-16 19:00:05  2025-05-16    19                   VRT 1   
2   2025-05-16 19:46:40  2025-05-16    19                   VRT 1   
3   2025-05-16 20:43:22  2025-05-16    20                     VTM   
4   2025-05-16 20:08:26  2025-05-16    20                     VTM   
..                  ...         ...   ...                     ...   
195 2025-05-25 18:20:00  2025-05-25    18                     VTM   
196 2025-05-25 20:00:14  2025-05-25    20              VRT CANVAS   
197 2025-05-25 18:26:59  2025-05-25    18  DAZN PRO LEAGUE 1 (NL)   
198 2025-05-25 16:57:40  2025-05-25    16        EUROSPORT 1 (NL)   
199 2025-05-25 20:00:03  2025-05-25    20                   PLAY4   

                                             Programma  Lengte_sec  Kijkers  \
0                                                THUIS        1489   937648   
1            

In [10]:
teVoorspellenData = pd.merge(teVoorspellenClean, histWeerdataClean, on=['date', 'hour', 'minute'], how='left')
teVoorspellenData = teVoorspellenData.drop(columns=['datetime', 'minute'])
print(teVoorspellenData)

             FullDate        date  hour Kanaal           Programma  \
0 2025-06-06 20:17:52  2025-06-06    20  VRT 1               THUIS   
1 2025-06-06 19:00:04  2025-06-06    19  VRT 1  HET 7 UUR-JOURNAAL   
2 2025-06-06 18:29:07  2025-06-06    18  VRT 1             BLOKKEN   
3 2025-06-06 18:59:48  2025-06-06    18    VTM      NIEUWS 19U VTM   
4 2025-06-06 21:37:08  2025-06-06    21  VRT 1  DE DAG VAN VANDAAG   
5 2025-06-06 20:05:28  2025-06-06    20    VTM             FAMILIE   
6 2025-06-06 20:41:26  2025-06-06    20    VTM        HUIS GEMAAKT   
7 2025-06-06 13:00:04  2025-06-06    13  VRT 1  HET 1 UUR-JOURNAAL   

   Lengte_sec Kijkers  Temperatuur  Gevoelstemp  Regen  Sneeuw  Weercode  \
0        1470    None         17.1         15.7    0.1     0.0      51.0   
1        2598    None         17.6         16.1    0.1     0.0      51.0   
2        1710    None         18.0         16.3    0.1     0.0      51.0   
3        3201    None         18.0         16.3    0.1     0.0   

In [11]:
def tijdFeatures(df):
    df['date'] = pd.to_datetime(df['date'])
    #feestdagen
    feestdagen = holidays.BE()
    df['isFeestdag'] = df['date'].apply(lambda x: 1 if x in feestdagen else 0)
    #dag van de week
    df['Weekdag'] = df['date'].dt.weekday
    #weekend
    df['isWeekend'] = df['Weekdag'].apply(lambda x: 1 if x >= 5 else 0)
    #seizoenen
    df['Seizoen'] = df['date'].apply(seizoenFinder)

    return df

#seizoen
def seizoenFinder(datum):
    inputDatum = datum.date()
    Y = inputDatum.year
    seizoenen = {
        'lente': (date(Y, 3, 20), date(Y, 6, 20)),
        'zomer': (date(Y, 6, 21), date(Y, 9, 22)),
        'herfst':   (date(Y, 9, 23), date(Y, 12, 20)),
        'winter': (date(Y, 12, 21), date(Y + 1, 3, 19)),
    }

    for seizoen, (start, end) in seizoenen.items():
        if start <= inputDatum <= end:
            return seizoen
    return 'winter'

In [12]:
teVoorspellenData = tijdFeatures(teVoorspellenData)
histKijkcijfersWeer = tijdFeatures(histKijkcijfersWeer)
histWeerdataClean = tijdFeatures(histWeerdataClean)
histWeerdataClean.drop(columns=['minute'], inplace=True)
display(teVoorspellenData.head())
display(histKijkcijfersWeer.head())
display(histWeerdataClean.head())

,FullDate,date,hour,Kanaal,Programma,Lengte_sec,Kijkers,Temperatuur,Gevoelstemp,Regen,Sneeuw,Weercode,Bewolking,Windsnelheid,Zonnenschijn,isFeestdag,Weekdag,isWeekend,Seizoen
0,2025-06-06 20:17:52,2025-06-06,20,VRT 1,THUIS,1470,None,17.1,15.7,0.1,0.0,51.0,85.0,9.9,3600.0,0,4,0,lente
1,2025-06-06 19:00:04,2025-06-06,19,VRT 1,HET 7 UUR-JOURNAAL,2598,None,17.6,16.1,0.1,0.0,51.0,77.0,10.9,3600.0,0,4,0,lente
2,2025-06-06 18:29:07,2025-06-06,18,VRT 1,BLOKKEN,1710,None,18.0,16.3,0.1,0.0,51.0,65.0,12.8,3600.0,0,4,0,lente
3,2025-06-06 18:59:48,2025-06-06,18,VTM,NIEUWS 19U VTM,3201,None,18.0,16.3,0.1,0.0,51.0,65.0,12.8,3600.0,0,4,0,lente
4,2025-06-06 21:37:08,2025-06-06,21,VRT 1,DE DAG VAN VANDAAG,3206,None,16.4,15.0,0.0,0.0,3.0,84.0,9.3,3600.0,0,4,0,lente


,FullDate,date,hour,Kanaal,Programma,Lengte_sec,Kijkers,Temperatuur,Gevoelstemp,Regen,Sneeuw,Weercode,Bewolking,Windsnelheid,Zonnenschijn,isFeestdag,Weekdag,isWeekend,Seizoen
0,2025-05-16 20:15:38,2025-05-16,20,VRT 1,THUIS,1489,937648,12.3,8.6,0.0,0.0,3.0,100.0,18.4,0.00,0,4,0,lente
1,2025-05-16 19:00:05,2025-05-16,19,VRT 1,HET 7 UUR-JOURNAAL,2685,760119,12.6,9.0,0.0,0.0,3.0,100.0,18.8,1895.07,0,4,0,lente
2,2025-05-16 19:46:40,2025-05-16,19,VRT 1,IEDEREEN BEROEMD,1331,581626,12.6,9.0,0.0,0.0,3.0,100.0,18.8,1895.07,0,4,0,lente
3,2025-05-16 20:43:22,2025-05-16,20,VTM,I CAN SEE YOUR VOICE,5500,527119,12.3,8.6,0.0,0.0,3.0,100.0,18.4,0.00,0,4,0,lente
4,2025-05-16 20:08:26,2025-05-16,20,VTM,FAMILIE,1468,526533,12.3,8.6,0.0,0.0,3.0,100.0,18.4,0.00,0,4,0,lente


,datetime,date,hour,Temperatuur,Gevoelstemp,Regen,Sneeuw,Weercode,Bewolking,Windsnelheid,Zonnenschijn,isFeestdag,Weekdag,isWeekend,Seizoen
0,2025-05-16 00:00:00,2025-05-16,0,11.6,8.6,0.0,0.0,0.0,0.0,12.7,0.0,0,4,0,lente
1,2025-05-16 01:00:00,2025-05-16,1,11.1,8.0,0.0,0.0,0.0,0.0,12.1,0.0,0,4,0,lente
2,2025-05-16 02:00:00,2025-05-16,2,10.3,7.2,0.0,0.0,0.0,0.0,11.5,0.0,0,4,0,lente
3,2025-05-16 03:00:00,2025-05-16,3,9.1,6.2,0.0,0.0,0.0,0.0,10.5,0.0,0,4,0,lente
4,2025-05-16 04:00:00,2025-05-16,4,8.4,5.7,0.0,0.0,0.0,0.0,10.0,0.0,0,4,0,lente


In [13]:
def lagFeatures(df, hist):
  df['Kijkers'] = None
  df['teVoorspellen'] = True
  hist['teVoorspellen'] = False
  df = pd.concat([hist, df], ignore_index=True)
  # Sorteer op tijd binnen elke groep
  df = df.sort_values(['Programma', 'FullDate'])

  df = df.sort_values(['Programma', 'FullDate'])

  # Bereken gemiddelde kijkers per Programma (op basis van historische data)
  kijkers_mean = df.groupby('Programma')['Kijkers'].transform('mean')

  # Bereken lag features per programma
  for i in range(1, 4):
      df[f'Kijkers_lag_{i}'] = df.groupby('Programma')['Kijkers'].shift(i)
      df[f'Kijkers_lag_{i}'] = df[f'Kijkers_lag_{i}'].fillna(df.groupby('Programma')['Kijkers'].transform('mean'))

  return df

In [14]:
pred_hist_df = lagFeatures(teVoorspellenData, histKijkcijfersWeer)
print("lagfeatures toegevoegd: ")
# display(pred_hist_df.dtypes)
pred_hist_df = pred_hist_df[pred_hist_df['teVoorspellen']]
pred_hist_df.drop(columns=['teVoorspellen'], inplace=True)
display(pred_hist_df.tail(10))
print(pred_hist_df[pred_hist_df.isna().any(axis=1)])



lagfeatures toegevoegd: 


C:\Users\krist\AppData\Local\Temp\ipykernel_20168\3750620475.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[f'Kijkers_lag_{i}'] = df[f'Kijkers_lag_{i}'].fillna(df.groupby('Programma')['Kijkers'].transform('mean'))
C:\Users\krist\AppData\Local\Temp\ipykernel_20168\3750620475.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[f'Kijkers_lag_{i}'] = df[f'Kijkers_lag_{i}'].fillna(df.groupby('Programma')['Kijkers'].transform('mean'))
C:\Users\krist\AppData\Local\Temp\ipykernel_20168\3750620475.py:17: FutureWarning: Downcasting object dtype arrays

,FullDate,date,hour,Kanaal,Programma,Lengte_sec,Kijkers,Temperatuur,Gevoelstemp,Regen,...,Bewolking,Windsnelheid,Zonnenschijn,isFeestdag,Weekdag,isWeekend,Seizoen,Kijkers_lag_1,Kijkers_lag_2,Kijkers_lag_3
202,2025-06-06 18:29:07,2025-06-06,18,VRT 1,BLOKKEN,1710,None,18.0,16.3,0.1,...,65.0,12.8,3600.00,0,4,0,lente,498063.0,488115.0,521698.0
204,2025-06-06 21:37:08,2025-06-06,21,VRT 1,DE DAG VAN VANDAAG,3206,None,16.4,15.0,0.0,...,84.0,9.3,3600.00,0,4,0,lente,NaN,NaN,NaN
205,2025-06-06 20:05:28,2025-06-06,20,VTM,FAMILIE,1544,None,17.1,15.7,0.1,...,85.0,9.9,3600.00,0,4,0,lente,394309.0,439141.0,465875.0
207,2025-06-06 13:00:04,2025-06-06,13,VRT 1,HET 1 UUR-JOURNAAL,1653,None,17.9,15.8,0.6,...,32.0,18.6,37.35,0,4,0,lente,413897.0,345994.0,324163.0
201,2025-06-06 19:00:04,2025-06-06,19,VRT 1,HET 7 UUR-JOURNAAL,2598,None,17.6,16.1,0.1,...,77.0,10.9,3600.00,0,4,0,lente,743261.0,623399.0,729945.0
206,2025-06-06 20:41:26,2025-06-06,20,VTM,HUIS GEMAAKT,3945,None,17.1,15.7,0.1,...,85.0,9.9,3600.00,0,4,0,lente,459562.0,400905.0,430233.5
203,2025-06-06 18:59:48,2025-06-06,18,VTM,NIEUWS 19U VTM,3201,None,18.0,16.3,0.1,...,65.0,12.8,3600.00,0,4,0,lente,472644.0,478956.0,454132.0
200,2025-06-06 20:17:52,2025-06-06,20,VRT 1,THUIS,1470,None,17.1,15.7,0.1,...,85.0,9.9,3600.00,0,4,0,lente,786790.0,798657.0,806473.0


               FullDate       date  hour Kanaal           Programma  \
202 2025-06-06 18:29:07 2025-06-06    18  VRT 1             BLOKKEN   
204 2025-06-06 21:37:08 2025-06-06    21  VRT 1  DE DAG VAN VANDAAG   
205 2025-06-06 20:05:28 2025-06-06    20    VTM             FAMILIE   
207 2025-06-06 13:00:04 2025-06-06    13  VRT 1  HET 1 UUR-JOURNAAL   
201 2025-06-06 19:00:04 2025-06-06    19  VRT 1  HET 7 UUR-JOURNAAL   
206 2025-06-06 20:41:26 2025-06-06    20    VTM        HUIS GEMAAKT   
203 2025-06-06 18:59:48 2025-06-06    18    VTM      NIEUWS 19U VTM   
200 2025-06-06 20:17:52 2025-06-06    20  VRT 1               THUIS   

     Lengte_sec Kijkers  Temperatuur  Gevoelstemp  Regen  ...  Bewolking  \
202        1710    None         18.0         16.3    0.1  ...       65.0   
204        3206    None         16.4         15.0    0.0  ...       84.0   
205        1544    None         17.1         15.7    0.1  ...       85.0   
207        1653    None         17.9         15.8    0.6

In [15]:
def oneHot(df):
  with open('./models/oneHotEncoder.pkl', 'rb') as oneHotFile:
    oneHotEnc = pickle.load(oneHotFile)

  lageKard = df[[ 'hour','Kanaal', 'isFeestdag', 'Weekdag', 'Seizoen']]
  dfOneHot = oneHotEnc.transform(lageKard)

  oneHotOutp = pd.DataFrame(dfOneHot.toarray(), 
                            columns=oneHotEnc.get_feature_names_out(), 
                            index=lageKard.index)

  df = df.drop(columns=['hour', 'Kanaal', 'isFeestdag', 'Weekdag', 'Seizoen'])
  df = pd.concat([df, oneHotOutp], axis = 1)
  return df

def target(df):
  #target encoding voor medium kardinaliteiten
  with open('./models/oneHotTarget.pkl', 'rb') as f:
    targetEnc = pickle.load(f)
  medKardinaliteit = df[['date', 'Programma', 'Lengte_sec', 'Temperatuur', 'Gevoelstemp', 'Regen', 'Bewolking', 'Windsnelheid', 'Zonnenschijn']]
  #verdere feature engineering op vorig model
  target = targetEnc.transform(medKardinaliteit)
  df = df.drop(columns=['date', 'Programma', 'Lengte_sec', 'Temperatuur', 'Gevoelstemp', 'Regen', 'Bewolking', 'Windsnelheid', 'Zonnenschijn'])
  f = pd.concat([df, target], axis=1)

  return f

In [16]:
teVoorspellenData = oneHot(pred_hist_df)
print("onehotencoding: ")
display(teVoorspellenData.head())
# Target encoding
targetOneHotEnc = target(teVoorspellenData)
print("targetencoding: ")
display(targetOneHotEnc.head())

onehotencoding: 


,FullDate,date,Programma,Lengte_sec,Kijkers,Temperatuur,Gevoelstemp,Regen,Sneeuw,Weercode,...,Weekdag_1,Weekdag_2,Weekdag_3,Weekdag_4,Weekdag_5,Weekdag_6,Seizoen_herfst,Seizoen_lente,Seizoen_winter,Seizoen_zomer
202,2025-06-06 18:29:07,2025-06-06,BLOKKEN,1710,None,18.0,16.3,0.1,0.0,51.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
204,2025-06-06 21:37:08,2025-06-06,DE DAG VAN VANDAAG,3206,None,16.4,15.0,0.0,0.0,3.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
205,2025-06-06 20:05:28,2025-06-06,FAMILIE,1544,None,17.1,15.7,0.1,0.0,51.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
207,2025-06-06 13:00:04,2025-06-06,HET 1 UUR-JOURNAAL,1653,None,17.9,15.8,0.6,0.0,53.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
201,2025-06-06 19:00:04,2025-06-06,HET 7 UUR-JOURNAAL,2598,None,17.6,16.1,0.1,0.0,51.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0


targetencoding: 


,FullDate,Kijkers,Sneeuw,Weercode,isWeekend,Kijkers_lag_1,Kijkers_lag_2,Kijkers_lag_3,hour_0,hour_1,...,Seizoen_zomer,date,Programma,Lengte_sec,Temperatuur,Gevoelstemp,Regen,Bewolking,Windsnelheid,Zonnenschijn
202,2025-06-06 18:29:07,None,0.0,51.0,0,498063.0,488115.0,521698.0,0.0,0.0,...,0.0,2025-06-06,589.000000,1710,18.0,16.3,0.1,65.0,12.8,3600.00
204,2025-06-06 21:37:08,None,0.0,3.0,0,NaN,NaN,NaN,0.0,0.0,...,0.0,2025-06-06,920.032052,3206,16.4,15.0,0.0,84.0,9.3,3600.00
205,2025-06-06 20:05:28,None,0.0,51.0,0,394309.0,439141.0,465875.0,0.0,0.0,...,0.0,2025-06-06,1403.000000,1544,17.1,15.7,0.1,85.0,9.9,3600.00
207,2025-06-06 13:00:04,None,0.0,53.0,0,413897.0,345994.0,324163.0,0.0,0.0,...,0.0,2025-06-06,1676.000000,1653,17.9,15.8,0.6,32.0,18.6,37.35
201,2025-06-06 19:00:04,None,0.0,51.0,0,743261.0,623399.0,729945.0,0.0,0.0,...,0.0,2025-06-06,1677.000000,2598,17.6,16.1,0.1,77.0,10.9,3600.00


In [17]:
teVoorspellenProgramma = teVoorspellenData['Programma']
teVoorspellenData = targetOneHotEnc.select_dtypes(include=[np.number])
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
teVoorspellenData.dtypes
display(teVoorspellenData)

,Sneeuw,Weercode,isWeekend,Kijkers_lag_1,Kijkers_lag_2,Kijkers_lag_3,hour_0,hour_1,hour_2,hour_6,hour_7,hour_8,hour_9,hour_10,hour_11,hour_12,hour_13,hour_14,hour_15,hour_16,hour_17,hour_18,hour_19,hour_20,hour_21,hour_22,hour_23,Kanaal_AB3,Kanaal_CANVAS,Kanaal_CAZ,Kanaal_Canvas,Kanaal_DAZN_PRO_LEAGUE_1_(NL),Kanaal_EEN,Kanaal_ELEVEN_PRO_LEAGUE_1_NL,Kanaal_EUROSPORT_1_(NL),Kanaal_KETNET,Kanaal_LA_UNE,Kanaal_OP_12,Kanaal_PLAY4,Kanaal_PLAY5,Kanaal_PLAY6,Kanaal_PLAY_SPORTS_OPEN,Kanaal_Q2,Kanaal_RTL-TVI,Kanaal_TF1,Kanaal_VIER,Kanaal_VIJF,Kanaal_VITAYA,Kanaal_VRT_1,Kanaal_VRT_CANVAS,Kanaal_VTM,Kanaal_VTM2,Kanaal_VTM3,Kanaal_VTM4,Kanaal_VTM_GOLD,Kanaal_ZES,isFeestdag_0,isFeestdag_1,Weekdag_0,Weekdag_1,Weekdag_2,Weekdag_3,Weekdag_4,Weekdag_5,Weekdag_6,Seizoen_herfst,Seizoen_lente,Seizoen_winter,Seizoen_zomer,Programma,Lengte_sec,Temperatuur,Gevoelstemp,Regen,Bewolking,Windsnelheid,Zonnenschijn
202,0.0,51.0,0,498063.0,488115.0,521698.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,589.000000,1710,18.0,16.3,0.1,65.0,12.8,3600.00
204,0.0,3.0,0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,920.032052,3206,16.4,15.0,0.0,84.0,9.3,3600.00
205,0.0,51.0,0,394309.0,439141.0,465875.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1403.000000,1544,17.1,15.7,0.1,85.0,9.9,3600.00
207,0.0,53.0,0,413897.0,345994.0,324163.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1676.000000,1653,17.9,15.8,0.6,32.0,18.6,37.35
201,0.0,51.0,0,743261.0,623399.0,729945.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1677.000000,2598,17.6,16.1,0.1,77.0,10.9,3600.00
206,0.0,51.0,0,459562.0,400905.0,430233.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1853.295495,3945,17.1,15.7,0.1,85.0,9.9,3600.00
203,0.0,51.0,0,472644.0,478956.0,454132.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,2553.000000,3201,18.0,16.3,0.1,65.0,12.8,3600.00
200,0.0,51.0,0,786790.0,798657.0,806473.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,3786.000000,1470,17.1,15.7,0.1,85.0,9.9,3600.00


In [20]:
try:
    with open('./models/optunaBestModel.pkl', 'rb') as file:
        lgbm = pickle.load(file)
except Exception as e:
    print("Kon model niet laden:", e)
    exit(1)

predictions = lgbm.predict(teVoorspellenData)
predictions = np.round(predictions).astype(int)
resultaten = pd.DataFrame({
    'Predicted': predictions,
    'Programma': teVoorspellenProgramma,
})
display(resultaten)

,Predicted,Programma
202,456969,BLOKKEN
204,114918,DE DAG VAN VANDAAG
205,421794,FAMILIE
207,342148,HET 1 UUR-JOURNAAL
201,653613,HET 7 UUR-JOURNAAL
206,424696,HUIS GEMAAKT
203,475103,NIEUWS 19U VTM
200,683309,THUIS
